# Price optimisation

### We have a set of risk bins, 1,2,3,4,5. We also have segmentations on the population. We have an overall rate increase target for renewals, maybe one for new business too but separate. We may also have some constraints on the price like, all segments inside risk bin 1 have the lowest price and is negative. We can also have constraints on specific segments and also orderings too, say if you want S_1 rate increase to be greater than S_2 - you can set that.

### We can solve this using an Integer programming problem, we specify $n$ specific price points $p_1,.....,p_n$. A vector $x_s = [1,0,0,0,...]$ for segment s means that segment s has rate increase $p_1$.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import seaborn as sns
sns.set_style("darkgrid")
sns.set(rc={'figure.figsize':(12,10)})
pd.set_option('display.max_columns',200)
sns.set(font_scale=1.75)
myid = os.getlogin()

sys.path.insert(0,r"../rebuild_notebooks")

from fiq_rebuild_p1 import importNewRun,returnPseudoValenBin


0. Import data and create risk bins and segments

In [2]:
df = importNewRun(version="v4",fname='dfyy_t_full_36k_v4_fs',YHAT='q0.690')

In [3]:
# risk bin
def returnRiskBintiles(df_in):
    cum_prop = [0.075,0.325,0.675,0.925,1.00]
    nbins = len(cum_prop)
    d = df_in.copy()
    d.sort_values(by='predicted_loss_ratio',inplace=True)
    d.reset_index(inplace=True)
    d["count_index"]= d.index.map(lambda x: (x+1)/len(d))
    d["risk_bin"] = d["count_index"].map(lambda x: returnPseudoValenBin(x,cum_prop,nbins))
    return d[["risk_bin","predicted_loss_ratio"]].groupby(by="risk_bin",as_index=False).max()

def getBin(val,binTable):
    for i,x in binTable.iterrows():
        if val <= x["predicted_loss_ratio"]:
            return i+1

myBintiles = returnRiskBintiles(df)
df["risk_bin"] = df["predicted_loss_ratio"].map(lambda r: getBin(r,myBintiles))

In [4]:
def segsize(l):
    if l <= 5:
        return "small"
    elif l <= 15:
        return "medium"
    else:
        return "large"

df["segment_state"] = df["insured_state"].map(lambda st: st if st in ["ca","tx"] else "other")
df["segment_longhaul"] = df["radius_op_r2_zone_long_haul_prp"].map(lambda r: "long" if r > 0.5 else "short")
df["segment_accountsize"] = df["powerunit_ct"].map(lambda r: segsize(r))



In [5]:
full_data = df.groupby(by=["segment_state","segment_longhaul"],as_index=False).agg(
{"policy_id":"count",
 "prem_i_total":"sum",
 "incurred_iu_sum":"sum"
}
)
segment_data = full_data.reset_index().rename(columns={"index":"segment_id"})

In [6]:
segment_data["loss_ratio"] = segment_data["incurred_iu_sum"]/segment_data["prem_i_total"]
segment_data

,segment_id,segment_state,segment_longhaul,policy_id,prem_i_total,incurred_iu_sum,loss_ratio
0,0,ca,long,82,3.946858e+06,1.170442e+06,0.296550
1,1,ca,short,1349,4.275390e+07,2.614342e+07,0.611486
2,2,other,long,219,1.419074e+07,9.897782e+06,0.697482
3,3,other,short,2148,9.810800e+07,4.574502e+07,0.466272
4,4,tx,long,215,1.512300e+07,1.411486e+07,0.933337
5,5,tx,short,847,6.160937e+07,4.614309e+07,0.748962


In [7]:
ddf = pd.merge(df,segment_data[["segment_id","segment_state","segment_longhaul"]],on=["segment_state","segment_longhaul"],how='left')

In [8]:
prem_df = pd.pivot_table(ddf,values="prem_i_total",index="segment_id",columns="risk_bin")
prem = prem_df.values

In [9]:
prem_df

risk_bin,1,2,3,4,5
segment_id,,,,,
0,20246.772295,24938.924135,55540.486945,92090.913274,NaN
1,13350.498680,19603.106017,34240.460308,61730.314881,77752.883696
2,23452.646760,25396.251601,59040.031605,104973.106104,156557.873336
3,14973.265375,19341.009014,39596.168816,66059.375563,131134.417336
4,9233.023278,11190.269678,32475.089150,78854.266437,152734.975959
5,14456.925289,12931.489081,46393.605749,73052.210604,168365.361736


In [10]:
prem

array([[ 20246.77229453,  24938.9241353 ,  55540.48694515,
         92090.91327372,             nan],
       [ 13350.49868037,  19603.10601704,  34240.4603083 ,
         61730.31488056,  77752.88369635],
       [ 23452.64675963,  25396.25160053,  59040.03160543,
        104973.10610377, 156557.873336  ],
       [ 14973.26537515,  19341.00901448,  39596.16881557,
         66059.37556284, 131134.41733585],
       [  9233.02327809,  11190.26967806,  32475.08915027,
         78854.26643706, 152734.97595906],
       [ 14456.92528889,  12931.48908114,  46393.60574902,
         73052.21060438, 168365.36173574]])

In [11]:
# risk segment 2 (first val) , risk bin 2 
prem[2,1]

25396.251600533786

Optimisation

In [12]:
import pyomo.environ as pyo

In [80]:
# Attempt 1: Maximise the difference in average price between risk segments
# create model
m = pyo.ConcreteModel()
# diff (minimum amount of difference required in price between green orange and red)
c_diff = 0.02
# diff (same segment, different risk bracket)
r_diff = 0.02
# max rate change
maxRate = 0.08
# min rate change
minRate = 0.07

Ic = [1,2,3,4]
I = [1,2,3,4,5]
J = [1,2,3,4,5,6]

# traffic light system
s = {}
s[1] = 'green'
s[2] = 'orange'
s[3] = 'orange'
s[4] = 'green'
s[5] = 'red'
s[6] = 'red'

# premium load
p = {}
for i in I:
    for j in J:
        p[i,j] = prem[j-1,i-1]
p[5,1]=100

In [84]:
# create model
m = pyo.ConcreteModel()
# Var
matrix_dims = (5, 6)  # row, col
m.xRisk = pyo.Set(initialize=range(1,matrix_dims[0]+1))
m.ySegment = pyo.Set(initialize=range(1,matrix_dims[1]+1))
m.x = pyo.Var(m.xRisk,m.ySegment,domain=pyo.Reals)
# param
m.p = pyo.Param(m.xRisk,m.ySegment,initialize=p,default=0)
m.Ic = Ic
m.I = I
m.J = J
# Maximise the difference in average price between risk segments
def ObjRule(m):
    return -1*sum(
            (sum((1.0+m.x[i+1,j]*m.p[i+1,j]) for j in m.J)/sum(m.p[i+1,j] for j in m.J)
               - sum((1.0+m.x[i,j])*m.p[i,j] for j in m.J)/sum(m.p[i,j] for j in m.J)
            )**2
               for i in m.Ic)
m.obj = pyo.Objective(rule=ObjRule,sense=pyo.maximize)
# CONSTRAINTS
m.cons = pyo.ConstraintList()
# 1 green < orange < red within the same risk segment by at least c_diff (e.g. 2%)
for i in I:
    for j in J:
        for k in J:
            if ((s[j] == "green") & (s[k] == "orange")):
                m.cons.add(m.x[i,j] + c_diff <= m.x[i,k])
            if ((s[j] == "orange") & (s[k] == "red")):
                m.cons.add(m.x[i,j] + c_diff <= m.x[i,k])
# 2 x[i,j] < x[i+1,j] < ... (within the same segment, if risk bin increases, the rate must increase by at least r_diff)
for j in J:
    for i in [1,2,3,4]:
        m.cons.add(m.x[i,j] + r_diff <= m.x[i+1,j])
# 3a Upper bound for overall rate change
m.cons.add( sum(sum((1+m.x[i,j])*p[i,j] for i in I) for j in J)/sum(sum(p[i,j] for i in I) for j in J) <= 1.0 + maxRate )
# 3b Lower bound for overall rate change
m.cons.add( sum(sum((1+m.x[i,j])*p[i,j] for i in I) for j in J)/sum(sum(p[i,j] for i in I) for j in J) >= 1.0 + minRate )
# Upper bound constraints
for j in J:
    m.cons.add(m.x[1,j] <= 0.0)
for i in I:
    for j in J:
        m.cons.add(m.x[i,j]<=0.25)
# Lower bouund constraints
for i in I:
    for j in J:
        if i > 1:
            m.cons.add(m.x[i,j]>=0.0)
        else:
            m.cons.add(m.x[i,j]>=-0.1)

In [85]:
solver = pyo.SolverFactory('ipopt')
solution = solver.solve(m)

In [86]:
results = {}
for i in I:
    results[i] = []
    for j in J:
        results[i].append(m.x[i,j]())

resdf = pd.DataFrame(results)
resdf

,1,2,3,4,5
0,-0.10,-7.934512e-09,0.02,0.04,0.075431
1,-0.08,2.000000e-02,0.04,0.06,0.128516
2,-0.08,1.999999e-02,0.04,0.06,0.122415
3,-0.10,-7.735200e-09,0.02,0.04,0.069623
4,-0.06,4.000002e-02,0.06,0.08,0.188610
5,-0.06,4.000002e-02,0.06,0.08,0.185657
